In [5]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler
from ols_diagnostics import full_diagnostic_report, validation_report

In [6]:
df = pd.read_csv('pullman_data.csv')
df.head()

,Date,DayName,OwnPrice (Rp),CompetitorPrice (Rp),Occupancy (%)
0,2025-01-01,Wednesday,1499179,1490434,99.4
1,2025-01-02,Thursday,1755757,1466462,73.0
2,2025-01-03,Friday,1769803,1716859,65.9
3,2025-01-04,Saturday,1812768,1660975,95.3
4,2025-01-05,Sunday,1722589,1226016,82.8


In [7]:
# 1. Load & Parsing Tanggal
df['Date'] = pd.to_datetime(df['Date'])

# 2. Extract Calendar Features
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

# 5. Tentukan Feature Set (Hindari Multikolinearitas)
feature_cols = [
    'OwnPrice (Rp)',
    'CompetitorPrice (Rp)',
    'Month'
]

X = df[feature_cols]
y = df['Occupancy (%)']

# Train-Test Split (273 data pertama = Jan - Sep 2025)
X_train = X.iloc[0:273]
X_test  = X.iloc[273:]
y_train = y.iloc[0:273]
y_test  = y.iloc[273:]

print(X_train.shape, X_test.shape)

(273, 3) (92, 3)


In [8]:
# X_train sudah dalam bentuk DataFrame numerik (tanpa konstanta, itu ditambah otomatis)
model, results = full_diagnostic_report(
    X_train, y_train,
    feature_names=feature_cols,
    y_name="Occupancy",
    model_label="Model 1 (OwnPrice, CompetitorPrice, Month)"
)

# Validasi ke data Okt-Des
val = validation_report(model, X_test, y_test, label="Okt-Des 2025")


######################################################################
#  FULL DIAGNOSTIC REPORT - Model 1 (OwnPrice, CompetitorPrice, Month)
######################################################################

1. PEARSON CORRELATION MATRIX (antar variabel independen)
                      OwnPrice (Rp)  CompetitorPrice (Rp)  Month
OwnPrice (Rp)                 1.000                 0.429  0.042
CompetitorPrice (Rp)          0.429                 1.000  0.115
Month                         0.042                 0.115  1.000

--- Korelasi tiap X terhadap Y (Occupancy) ---
            Variabel      r  p-value Sig
       OwnPrice (Rp) 0.2582   0.0000 ***
CompetitorPrice (Rp) 0.4746   0.0000 ***
               Month 0.0335   0.5812    

2. MODEL SUMMARY (Coefficients, t-test, F-test)
                        Coef  Std Error  t-stat  p-value  Sig
const                -0.9419     8.3769 -0.1124   0.9106     
OwnPrice (Rp)         0.0000     0.0000  1.1274   0.2606     
CompetitorPrice (Rp)

c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)


In [9]:
import contextlib

with open("model1_report.txt", "w") as f:
    with contextlib.redirect_stdout(f):
        model, results = full_diagnostic_report(
            X_train, y_train,
            feature_names=feature_cols,
            y_name="Occupancy",
            model_label="Model 1 (OwnPrice, CompetitorPrice, Month)"
        )
        val = validation_report(model, X_test, y_test, label="Okt-Des 2025")

print("Report tersimpan ke model1_report.txt")

Report tersimpan ke model1_report.txt


c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)
